[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/13_Q_Learning_Reinforcement_Learning.ipynb)

# Q-Learning — Reinforcement Learning from Scratch (GridWorld)

**Problem type:** Reinforcement Learning — an agent learns a *policy* from rewards, **NOT** from a labeled dataset.

---

## What is Reinforcement Learning?

In supervised learning you hand the algorithm a dataset of `(input, correct_answer)` pairs and it learns to predict.  
In unsupervised learning you provide data and let the algorithm find hidden structure.  
**Reinforcement learning is different: there is no dataset.** Instead, an **agent** interacts with an **environment** and learns purely by trial and error.

### Key concepts

| Concept | Meaning |
|---|---|
| **Agent** | The learner / decision-maker (our algorithm) |
| **Environment** | The world the agent lives in (the GridWorld) |
| **State `s`** | A snapshot of the situation (which cell the agent is in) |
| **Action `a`** | What the agent can do (up / down / left / right) |
| **Reward `r`** | A scalar signal the environment gives back (+10 for goal, -10 for trap, -1 per step) |
| **Policy `π`** | The agent's strategy: which action to take in each state |

### The Q-Table

`Q(s, a)` is a table that stores *how good* it is to take action `a` in state `s`.  
After training, the agent picks the action with the highest Q-value: `π(s) = argmax_a Q(s, a)`.

### Bellman Update (the learning rule)

After each step the agent updates the Q-table:

```
Q(s, a) ← Q(s, a) + α · [ r + γ · max_a' Q(s', a')  −  Q(s, a) ]
                           └─────────── TD target ──────────────┘
```

- **α (alpha)** — learning rate (how quickly to overwrite old estimates)
- **γ (gamma)** — discount factor (how much future rewards are worth now)
- **r + γ·max Q(s',a')** — the *target*: immediate reward + discounted best future value
- The difference in brackets is the *temporal-difference (TD) error*

### Exploration vs Exploitation (ε-greedy)

If the agent always picks the action it *thinks* is best, it may never discover better paths.  
With **ε-greedy**: with probability ε pick a *random* action (explore), otherwise pick the best known action (exploit).  
We decay ε over time so the agent explores heavily at first, then exploits what it has learned.

---

> **Summary of differences from supervised/unsupervised learning:**  
> - No labeled data, no feature matrix — the agent *generates* its own experience.  
> - The 'ground truth' is the reward signal, which is often sparse and delayed.  
> - Learning is sequential and online: each action changes the next state.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (works everywhere)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

np.random.seed(42)
print('Libraries loaded. NumPy:', np.__version__)


## 1. GridWorld Environment

A **5×5 grid** with:
- **Start** (top-left, cell 0): the agent always begins here
- **Goal** (bottom-right, cell 24): reward **+10**, episode ends
- **Traps** (cells 7 and 17): reward **-10**, episode ends
- **Step penalty**: **-1** for every move (encourages finding the *shortest* path)
- **Actions**: 0=Up, 1=Down, 2=Left, 3=Right (walls are bounced off — agent stays put)


In [ ]:
class GridWorld:
    """
    A simple 5x5 GridWorld RL environment (no external libraries needed).

    Layout (row, col) → state index = row * ncols + col:
        S  .  .  .  .
        .  .  T  .  .
        .  .  .  .  .
        .  .  T  .  .
        .  .  .  .  G

    S=start (0), G=goal (24), T=trap (7, 17)
    """

    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # up/down/left/right
    ACTION_SYMBOLS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def __init__(self, nrows=5, ncols=5, goal=24, traps=None, step_penalty=-1,
                 goal_reward=10, trap_reward=-10):
        self.nrows = nrows
        self.ncols = ncols
        self.n_states = nrows * ncols
        self.n_actions = 4
        self.goal = goal
        self.traps = set(traps if traps is not None else [7, 17])
        self.step_penalty = step_penalty
        self.goal_reward = goal_reward
        self.trap_reward = trap_reward
        self.state = 0

    def reset(self):
        """Return start state."""
        self.state = 0
        return self.state

    def step(self, action):
        """
        Take an action and return (next_state, reward, done).
        Walls: if the move goes off-grid the agent stays in place.
        """
        row, col = divmod(self.state, self.ncols)
        dr, dc = self.ACTIONS[action]
        new_row = max(0, min(self.nrows - 1, row + dr))
        new_col = max(0, min(self.ncols - 1, col + dc))
        self.state = new_row * self.ncols + new_col

        if self.state == self.goal:
            return self.state, self.goal_reward, True
        if self.state in self.traps:
            return self.state, self.trap_reward, True
        return self.state, self.step_penalty, False

    def render_layout(self, ax=None):
        """Visualise the grid layout (static, no policy shown)."""
        show = ax is None
        if show:
            fig, ax = plt.subplots(figsize=(5, 5))
        grid = np.zeros((self.nrows, self.ncols))
        # colour codes: 0=empty, 1=trap, 2=goal, 3=start
        for t in self.traps:
            grid[divmod(t, self.ncols)] = 1
        grid[divmod(self.goal, self.ncols)] = 2
        grid[0, 0] = 3
        cmap = plt.cm.colors.ListedColormap(['#f0f0f0', '#e74c3c', '#2ecc71', '#3498db'])
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=3)
        for s in range(self.n_states):
            r, c = divmod(s, self.ncols)
            label = ''
            if s == 0:         label = 'START'
            elif s == self.goal: label = 'GOAL\n+10'
            elif s in self.traps: label = 'TRAP\n-10'
            ax.text(c, r, label, ha='center', va='center', fontsize=9, fontweight='bold')
        ax.set_xticks(np.arange(-0.5, self.ncols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.nrows, 1), minor=True)
        ax.grid(which='minor', color='gray', linewidth=1)
        ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
        ax.set_title('GridWorld Layout', fontsize=13, fontweight='bold')
        if show:
            plt.tight_layout()
            plt.savefig('/tmp/grid_layout.png', dpi=100, bbox_inches='tight')
            plt.show()
            print('Grid saved.')


env = GridWorld()
print(f'States: {env.n_states}, Actions: {env.n_actions}')
print(f'Goal state: {env.goal}, Trap states: {env.traps}')

# Visualise layout
env.render_layout()


## 2. Tabular Q-Learning

We train for **500 episodes** using ε-greedy with exponential epsilon decay.  
The Q-table starts at all zeros (no prior knowledge).


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
ALPHA   = 0.1    # learning rate
GAMMA   = 0.95   # discount factor
EPSILON = 1.0    # initial exploration rate
EPS_MIN = 0.01   # minimum epsilon
EPS_DECAY = 0.995  # multiplicative decay per episode
N_EPISODES = 500
MAX_STEPS  = 200  # safety cap per episode

# ── Q-table initialisation ────────────────────────────────────────────────────
Q = np.zeros((env.n_states, env.n_actions))  # shape (25, 4)

# ── Logging ───────────────────────────────────────────────────────────────────
episode_rewards = []
episode_steps   = []
epsilon = EPSILON

# ── Training loop ─────────────────────────────────────────────────────────────
np.random.seed(42)  # reproducibility

for ep in range(N_EPISODES):
    state = env.reset()
    total_reward = 0
    steps = 0

    for _ in range(MAX_STEPS):
        # ε-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.randint(env.n_actions)   # explore
        else:
            action = np.argmax(Q[state])                # exploit

        next_state, reward, done = env.step(action)

        # Bellman update
        td_target = reward + GAMMA * np.max(Q[next_state]) * (not done)
        td_error  = td_target - Q[state, action]
        Q[state, action] += ALPHA * td_error

        state = next_state
        total_reward += reward
        steps += 1

        if done:
            break

    # Decay epsilon
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    episode_rewards.append(total_reward)
    episode_steps.append(steps)

print(f'Training complete over {N_EPISODES} episodes.')
print(f'Last-50 avg reward : {np.mean(episode_rewards[-50:]):.2f}')
print(f'Last-50 avg steps  : {np.mean(episode_steps[-50:]):.1f}')
print(f'Final epsilon      : {epsilon:.4f}')


## 3. Learning Curves

We plot the **total reward per episode** (smoothed with a 20-episode rolling average)  
and the **steps to goal**. A good learning signal shows reward rising and stabilising, steps falling.


In [ ]:
def smooth(x, window=20):
    """Simple moving average for plotting."""
    if len(x) < window:
        return np.array(x)
    return np.convolve(x, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Reward per episode ────────────────────────────────────────────────────────
ax = axes[0]
episodes = np.arange(1, N_EPISODES + 1)
ax.plot(episodes, episode_rewards, alpha=0.25, color='steelblue', label='raw')
smoothed_r = smooth(episode_rewards, 20)
ax.plot(np.arange(20, N_EPISODES + 1), smoothed_r, color='steelblue', linewidth=2, label='smoothed (20-ep)')
ax.axhline(np.mean(episode_rewards[-50:]), color='orange', linestyle='--', linewidth=1.5,
           label=f'last-50 avg = {np.mean(episode_rewards[-50:]):.1f}')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Learning Curve: Total Reward per Episode')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Steps per episode ─────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(episodes, episode_steps, alpha=0.25, color='tomato', label='raw')
smoothed_s = smooth(episode_steps, 20)
ax.plot(np.arange(20, N_EPISODES + 1), smoothed_s, color='tomato', linewidth=2, label='smoothed (20-ep)')
ax.axhline(np.mean(episode_steps[-50:]), color='orange', linestyle='--', linewidth=1.5,
           label=f'last-50 avg = {np.mean(episode_steps[-50:]):.1f} steps')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps')
ax.set_title('Steps per Episode (lower = better)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Learning curves saved.')


## 4. Learned Policy & State-Value Heatmap

**Left:** The greedy policy — arrows show the action with the highest Q-value in each cell.  
**Right:** The state-value heatmap — brighter cells are more valuable (`V(s) = max_a Q(s,a)`).


In [ ]:
greedy_actions = np.argmax(Q, axis=1)        # shape (25,)
state_values   = np.max(Q, axis=1)            # V(s) = max_a Q(s,a)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Greedy policy (arrows) ────────────────────────────────────────────────────
ax = axes[0]
bg = np.zeros((env.nrows, env.ncols))
for t in env.traps:
    bg[divmod(t, env.ncols)] = -1
bg[divmod(env.goal, env.ncols)] = 1

cmap2 = LinearSegmentedColormap.from_list('gw', ['#e74c3c', '#f5f5f5', '#2ecc71'])
ax.imshow(bg, cmap=cmap2, vmin=-1, vmax=1, alpha=0.4)

arrow_kw = dict(ha='center', va='center', fontsize=18, fontweight='bold')
for s in range(env.n_states):
    r, c = divmod(s, env.ncols)
    if s == env.goal:
        ax.text(c, r, '★', **arrow_kw, color='darkgreen')
    elif s in env.traps:
        ax.text(c, r, '✗', **arrow_kw, color='darkred')
    else:
        ax.text(c, r, env.ACTION_SYMBOLS[greedy_actions[s]], **arrow_kw, color='#2c3e50')

ax.set_xticks(np.arange(-0.5, env.ncols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, env.nrows, 1), minor=True)
ax.grid(which='minor', color='gray', linewidth=1)
ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
ax.set_title('Learned Greedy Policy (arrows)', fontsize=12, fontweight='bold')

# ── State-value heatmap ───────────────────────────────────────────────────────
ax = axes[1]
V_grid = state_values.reshape(env.nrows, env.ncols)
im = ax.imshow(V_grid, cmap='RdYlGn')
for s in range(env.n_states):
    r, c = divmod(s, env.ncols)
    ax.text(c, r, f'{state_values[s]:.1f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='V(s) = max_a Q(s,a)')
ax.set_xticks(np.arange(-0.5, env.ncols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, env.nrows, 1), minor=True)
ax.grid(which='minor', color='gray', linewidth=1)
ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
ax.set_title('State-Value Heatmap V(s)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/policy_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Policy and value map saved.')


## 5. Greedy Trajectory After Training

Let the trained agent (ε = 0, pure exploitation) navigate from start to goal.  
A good policy should reach the goal in **8 steps** (shortest Manhattan-distance path avoiding traps).


In [ ]:
def run_greedy(env, Q, max_steps=50):
    """Run one episode with the fully greedy policy and return the trajectory."""
    state = env.reset()
    path  = [state]
    total_reward = 0

    for _ in range(max_steps):
        action = np.argmax(Q[state])
        state, reward, done = env.step(action)
        path.append(state)
        total_reward += reward
        if done:
            break

    return path, total_reward


path, greedy_reward = run_greedy(env, Q)

print('Greedy trajectory (state indices):')
print(' → '.join(map(str, path)))
print(f'Steps taken   : {len(path) - 1}')
print(f'Total reward  : {greedy_reward}')
reached_goal = path[-1] == env.goal
print(f'Reached goal  : {reached_goal}')

# ── Visualise trajectory on the grid ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
visit_count = np.zeros(env.n_states)
for s in path:
    visit_count[s] += 1
vc_grid = visit_count.reshape(env.nrows, env.ncols)

ax.imshow(vc_grid, cmap='Blues', vmin=0, vmax=max(visit_count) + 1)
for s in range(env.n_states):
    r, c = divmod(s, env.ncols)
    label = ''
    if s == 0:          label = 'S'
    elif s == env.goal: label = 'G'
    elif s in env.traps: label = 'T'
    if s in path:
        idx = path.index(s)
        label = f'{label}({idx})' if label else f'{idx}'
    ax.text(c, r, label, ha='center', va='center', fontsize=9)

ax.set_xticks(np.arange(-0.5, env.ncols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, env.nrows, 1), minor=True)
ax.grid(which='minor', color='gray', linewidth=1)
ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
ax.set_title(f'Greedy Path: {len(path)-1} steps, reward={greedy_reward}', fontsize=11)

plt.tight_layout()
plt.savefig('/tmp/trajectory.png', dpi=100, bbox_inches='tight')
plt.show()
print('Trajectory saved.')


## Findings

| Metric | Value |
|---|---|
| Training episodes | 500 |
| Final epsilon | ~0.08 (mostly exploiting) |
| Last-50 avg reward | ~ +2 (well above random ≈ -∞) |
| Greedy steps to goal | 8 (optimal) |
| Optimal policy learned? | **Yes** |

**What we observed:**

1. **Reward increases and stabilises** — the learning curve rises from random negative values to a positive plateau, confirming the agent has learned a useful policy.
2. **Steps decrease** — the agent stops wandering; it heads straight for the goal.
3. **The learned policy avoids traps** — the greedy policy arrows steer the agent around the -10 trap cells.
4. **State values make sense** — cells near the goal have higher V(s); cells near traps have lower V(s).
5. **ε-decay balances exploration and exploitation** — early episodes explore widely (high ε); later ones exploit the learned policy (low ε).


## Try it yourself

Experiment with the following changes to deepen your understanding:

1. **Change the grid size** — try `GridWorld(nrows=8, ncols=8, goal=63)` and re-run training. Does the agent still converge?
2. **Move the traps** — add more traps or put one on the shortest path: `traps=[7, 11, 17]`. How does the policy change?
3. **Tune α (learning rate)** — try `ALPHA=0.5` (faster but noisier) vs `ALPHA=0.01` (slower but stable).
4. **Tune γ (discount factor)** — set `GAMMA=0.5` and observe: does the agent become more short-sighted?
5. **Change ε-decay** — use `EPS_DECAY=0.99` (slower decay → more exploration). How many episodes to convergence?
6. **Add stochastic transitions** — modify `step()` to move in a random direction 20% of the time. Does Q-learning still converge?
7. **Implement SARSA** — replace `np.max(Q[next_state])` in the update with `Q[next_state, next_action]` where `next_action` is chosen ε-greedily. Compare policies: SARSA is *on-policy* and usually more conservative near traps.
8. **Increase episodes** — run 2000 episodes. Does the final policy improve further?
